# Deliverable 2 - Formulación automática de modelos de optimización

Este notebook usa **Qwen/Qwen2.5-1.5B-Instruct** para convertir descripciones en lenguaje natural de problemas deterministas de programación lineal o programación lineal entera mixta en formulaciones matemáticas, sin resolverlas.

## Equipo (grupo 1)

- José Luis Erices
- Isidora Rivera
- Rayen Muñoz
- Sebastián Soto

## Instrucciones de ejecución

1. En Google Colab, ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución**.
2. Selecciona **T4 GPU** como acelerador de hardware.
3. Ejecuta las celdas en orden. La primera descarga instala las dependencias y la carga inicial del modelo puede tardar algunos minutos.

# 1. Configuración e instalación de dependencias

In [ ]:
%pip install -q transformers accelerate

# 2. Imports

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 3. Configuración del modelo

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_DTYPE = torch.float16

if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU T4 en Google Colab antes de continuar.")

print(f"GPU disponible: {torch.cuda.get_device_name(0)}")

# 4. Carga de Qwen2.5-1.5B-Instruct

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map="auto",
)
model.eval()

print("Modelo cargado correctamente.")

# 5. Inferencia con direct prompting

El siguiente caso se conserva como base para la inferencia. En esta etapa todavía no se implementa la intervención del Deliverable 2.

In [ ]:
problema_io = r"""
En la ciudad de Concepción se requiere construir 2 centros oftalmológicos para atender las consultas médicas de la población. Existen 3 lugares potenciales donde estos centros pueden ser ubicados: Barrio Norte, Los Carrera y Collao.

Debido al tamaño de la zona, la comuna se ha dividido en 5 sectores de demanda: Concepción Centro, Sector Lomas, Costanera, Barrio Universitario y Nonguén. La población de cada sector, expresada en miles de personas, es la siguiente: Concepción Centro tiene 30 mil habitantes, Sector Lomas 15 mil, Costanera 5 mil, Barrio Universitario 2 mil y Nonguén 10 mil habitantes.

Las distancias entre los lugares potenciales para construir los centros y los sectores de demanda se expresan en kilómetros. Desde Barrio Norte, las distancias son de 1,3 km a Concepción Centro, 0,5 km a Sector Lomas, 2,5 km a Costanera, 2,0 km a Barrio Universitario y 3,0 km a Nonguén.

Desde Los Carrera, las distancias son de 0,3 km a Concepción Centro, 2,0 km a Sector Lomas, 1,2 km a Costanera, 1,0 km a Barrio Universitario y 1,9 km a Nonguén.

Finalmente, desde Collao, las distancias son de 1,6 km a Concepción Centro, 2,3 km a Sector Lomas, 2,1 km a Costanera, 1,5 km a Barrio Universitario y 0,4 km a Nonguén.

Se debe formular un modelo matemático que permita determinar cuáles 2 de los 3 lugares potenciales deben seleccionarse para instalar los centros oftalmológicos y cómo asignar los sectores de demanda a estos centros, buscando minimizar la distancia de atención considerando la población de cada sector.

Para la formulación del modelo, se deben definir los conjuntos, parámetros y variables de decisión, plantear la función objetivo correspondiente e incorporar las restricciones necesarias para garantizar que cada sector sea atendido por un único centro, que los sectores solo puedan ser asignados a centros que hayan sido construidos y que se instalen exactamente 2 centros oftalmológicos.
"""

In [ ]:
messages = [{"role": "user", "content": problema_io}]
texto_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(texto_prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

tokens_generados = outputs[0, inputs["input_ids"].shape[1]:]
respuesta = tokenizer.decode(tokens_generados, skip_special_tokens=True)

print("--- RESPUESTA DEL MODELO ---")
print(respuesta)